# NB07 — Model Performance & Outcome Impact Analysis

## Two Layers of Product Health for AI Products

When evaluating an AI product like SmarterDx, you can't just look at the model in isolation. You need to understand two critical layers:

### 🎯 Product Experience Layer
**Question:** "Is the user actually engaging with the AI features?"

This layer measures adoption, workflow integration, and whether users even SEE the model's output. Metrics include:
- AI suggestion funnel (model fired → surfaced → accepted)
- Acceptance rate by segment
- Time to decision
- User engagement with AI features

**Why it matters:** A brilliant model is worthless if nobody uses it. If your AI suggestions are hidden, ignored, or surfaced too infrequently, it doesn't matter how accurate they are.

### 🧠 Model Performance Layer
**Question:** "Is the AI producing accurate, useful results?"

This layer measures the model itself: precision, recall, calibration, false positive rates. Metrics include:
- Precision: "Of all suggestions made, what % were correct?"
- Recall: "Of all things that should be flagged, what % did we catch?"
- Calibration: "When the model says it's 80% confident, is it actually right 80% of the time?"
- Performance by category and cohort

**Why it matters:** If the model is wrong 40% of the time, it's destroying value even if users love the UI. In healthcare (SmarterDx example), a false positive diagnosis can harm patients.

### The Tradeoff
You can have:
- **Great model performance + poor product experience** = Nobody uses the accurate AI
- **Great product experience + poor model performance** = Users love the UI but the AI is wrong
- **Both!** = Real value capture

This notebook measures both layers and shows you how to connect them to understand true business impact.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve, roc_curve, auc
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
COLORS = {'blue': '#2E86AB', 'orange': '#F18F01', 'green': '#2CA58D', 'red': '#E15554'}
OUTPUTS = '../data/outputs/nb07/'
import os
os.makedirs(OUTPUTS, exist_ok=True)

model_df = pd.read_csv('../data/inputs/model_performance_clean.csv', parse_dates=['interaction_date'])
activity_df = pd.read_csv('../data/inputs/activity_clean.csv', parse_dates=['signup_date', 'activity_date'])

print(f'Model data: {len(model_df):,} interactions, {model_df["user_id"].nunique():,} users')
print(f'Activity data: {len(activity_df):,} events, {activity_df["user_id"].nunique():,} users')
print(f'\nModel data columns: {model_df.columns.tolist()}')
print(f'Date range: {model_df["interaction_date"].min()} to {model_df["interaction_date"].max()}')

In [ ]:
# Quick overview of model interaction funnel
print('=== AI Suggestion Funnel Overview ===')
print(f'Total interactions: {len(model_df):,}')
print(f'Model fired: {model_df["model_fired"].sum():,} ({model_df["model_fired"].mean()*100:.1f}%)')
print(f'Suggestion surfaced: {model_df["suggestion_surfaced"].sum():,} ({model_df["suggestion_surfaced"].mean()*100:.1f}%)')
print(f'User accepted: {model_df["user_accepted"].sum():,} ({model_df["user_accepted"].mean()*100:.1f}%)')
print(f'Ground truth available: {model_df["ground_truth_correct"].notna().sum():,} ({model_df["ground_truth_correct"].notna().mean()*100:.1f}%)')
print(f'Correct suggestions: {model_df["ground_truth_correct"].sum():,}')

# Basic stats by plan type
print('\n=== By Plan Type ===')
print(model_df.groupby('plan_type')[['model_fired', 'suggestion_surfaced', 'user_accepted', 'ground_truth_correct']].agg(['sum', 'mean']))

# Suggestion categories
print('\n=== Suggestion Categories ===')
print(model_df['suggestion_category'].value_counts())

## Section 1: Product Experience Layer

Before we evaluate the model itself, we need to understand **whether users are even engaging with the AI features**. This is the product experience layer — it answers:

> **"Is the AI integrated into the user's workflow? Are users seeing it, trusting it, and acting on it?"**

The best model in the world is worthless if:
- Users never see the suggestions (low surfacing rate)
- Users see them but ignore them (low acceptance)
- Users don't understand them (high time-to-decision)

Let's measure the product experience layer with four key metrics:
1. **AI Suggestion Funnel** — Where does the pipeline break down?
2. **Acceptance Rate by Segment** — Who trusts the AI?
3. **Time-to-Decision** — How confident are users?
4. **Engagement by Plan Type** — Does pricing affect engagement?


In [ ]:
# AI Suggestion Funnel: interactions → model fired → surfaced → accepted
funnel_stages = {
    'Total Interactions': len(model_df),
    'Model Fired': model_df['model_fired'].sum(),
    'Suggestion Surfaced': model_df[model_df['model_fired']]['suggestion_surfaced'].sum(),
    'User Accepted': model_df[model_df['user_accepted']]['user_accepted'].sum()
}

# Calculate conversion rates
conversions = {}
previous = len(model_df)
for stage, count in funnel_stages.items():
    rate = (count / previous) * 100 if previous > 0 else 0
    conversions[stage] = rate
    previous = count
    print(f'{stage:25s}: {count:8,d}  ({rate:6.2f}%)')

# Visualize funnel
fig, ax = plt.subplots(figsize=(12, 5))
stages = list(funnel_stages.keys())
counts = list(funnel_stages.values())
colors_list = [COLORS['blue'], COLORS['blue'], COLORS['orange'], COLORS['green']]

bars = ax.barh(stages, counts, color=colors_list, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add labels
for i, (bar, count) in enumerate(zip(bars, counts)):
    rate = conversions[stages[i]]
    label_text = f'{count:,}\n({rate:.1f}%)'
    ax.text(count + max(counts)*0.02, bar.get_y() + bar.get_height()/2,
            label_text, va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Count', fontsize=12, fontweight='bold')
ax.set_title('AI Suggestion Funnel: From Model Fire to User Acceptance', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUTS}01_ai_funnel.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ Funnel visualization saved')

In [ ]:
# Acceptance rate by plan type
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By plan type
ax = axes[0, 0]
plan_acceptance = model_df.groupby('plan_type')['user_accepted'].agg(['sum', 'count'])
plan_acceptance['rate'] = plan_acceptance['sum'] / plan_acceptance['count'] * 100
plan_acceptance_rate = plan_acceptance['rate'].sort_values(ascending=False)
bars = ax.bar(plan_acceptance_rate.index, plan_acceptance_rate.values,
              color=[COLORS['blue'], COLORS['orange'], COLORS['green']], alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Acceptance Rate (%)', fontsize=11, fontweight='bold')
ax.set_title('Acceptance Rate by Plan Type', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, f'{height:.1f}%',
            ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# By suggestion category
ax = axes[0, 1]
cat_acceptance = model_df.groupby('suggestion_category')['user_accepted'].agg(['sum', 'count'])
cat_acceptance['rate'] = cat_acceptance['sum'] / cat_acceptance['count'] * 100
cat_acceptance_rate = cat_acceptance['rate'].sort_values(ascending=False)
bars = ax.bar(range(len(cat_acceptance_rate)), cat_acceptance_rate.values,
              color=COLORS['blue'], alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xticks(range(len(cat_acceptance_rate)))
ax.set_xticklabels(cat_acceptance_rate.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Acceptance Rate (%)', fontsize=11, fontweight='bold')
ax.set_title('Acceptance Rate by Suggestion Category', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, f'{height:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# By cohort
ax = axes[1, 0]
cohort_acceptance = model_df.groupby('signup_cohort')['user_accepted'].agg(['sum', 'count'])
cohort_acceptance['rate'] = cohort_acceptance['sum'] / cohort_acceptance['count'] * 100
cohort_acceptance_rate = cohort_acceptance['rate'].sort_index()
ax.plot(cohort_acceptance_rate.index, cohort_acceptance_rate.values,
        marker='o', linewidth=2.5, markersize=8, color=COLORS['blue'])
ax.fill_between(range(len(cohort_acceptance_rate)), cohort_acceptance_rate.values, alpha=0.3, color=COLORS['blue'])
ax.set_xlabel('Signup Cohort', fontsize=11, fontweight='bold')
ax.set_ylabel('Acceptance Rate (%)', fontsize=11, fontweight='bold')
ax.set_title('Acceptance Rate by Cohort (Newer Users)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)

# Plan type + suggestion category heatmap
ax = axes[1, 1]
pivot = model_df.groupby(['plan_type', 'suggestion_category'])['user_accepted'].mean() * 100
pivot_table = pivot.unstack()
sns.heatmap(pivot_table, annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label': 'Acceptance Rate (%)'}, ax=ax, linewidths=1)
ax.set_title('Acceptance Rate: Plan Type × Category', fontsize=12, fontweight='bold')
ax.set_xlabel('Suggestion Category', fontsize=11, fontweight='bold')
ax.set_ylabel('Plan Type', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUTS}02_acceptance_segments.png', dpi=300, bbox_inches='tight')
plt.show()

print('Acceptance rates:')
print(f'By plan: {plan_acceptance_rate.to_dict()}')
print(f'By category: {cat_acceptance_rate.to_dict()}')
print('✓ Acceptance rate visualization saved')

In [ ]:
# Time-to-decision analysis: accepted vs rejected
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter for valid time-to-decision values
valid_time_df = model_df[model_df['time_to_decision_sec'].notna()].copy()

# Distribution by acceptance
ax = axes[0]
accepted = valid_time_df[valid_time_df['user_accepted'] == True]['time_to_decision_sec']
rejected = valid_time_df[valid_time_df['user_accepted'] == False]['time_to_decision_sec']

ax.hist(accepted, bins=40, alpha=0.6, label=f'Accepted (n={len(accepted)})', color=COLORS['green'], edgecolor='black')
ax.hist(rejected, bins=40, alpha=0.6, label=f'Rejected (n={len(rejected)})', color=COLORS['red'], edgecolor='black')
ax.set_xlabel('Time to Decision (seconds)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Time-to-Decision: Accepted vs Rejected Suggestions', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Summary statistics
print('Time-to-Decision Statistics (seconds):')
print(f'Accepted:  mean={accepted.mean():.1f}s, median={accepted.median():.1f}s, std={accepted.std():.1f}s')
print(f'Rejected:  mean={rejected.mean():.1f}s, median={rejected.median():.1f}s, std={rejected.std():.1f}s')

# By category
ax = axes[1]
category_time = valid_time_df.groupby('suggestion_category')['time_to_decision_sec'].agg(['mean', 'std']).sort_values('mean')
x_pos = range(len(category_time))
ax.barh(x_pos, category_time['mean'], xerr=category_time['std'],
        color=COLORS['blue'], alpha=0.8, edgecolor='black', linewidth=1.5, capsize=5)
ax.set_yticks(x_pos)
ax.set_yticklabels(category_time.index, fontsize=10)
ax.set_xlabel('Mean Time to Decision (seconds)', fontsize=11, fontweight='bold')
ax.set_title('Average Decision Time by Suggestion Category', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}03_time_to_decision.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Time-to-decision visualization saved')

## Section 2: Model Performance Layer

Now we shift focus to the **model itself**. This section answers:

> **"When the model makes a suggestion, is it actually correct?"**

### Key Concepts

**Precision:** "Of all the suggestions the model made, what % were correct?"
- High precision = when the model speaks, it's usually right
- Low precision = the model fires a lot of false alarms
- **In healthcare (SmarterDx):** Low precision = many false diagnosis suggestions = alert fatigue

**Recall:** "Of all the things that SHOULD have been flagged, what % did the model catch?"
- High recall = the model finds most of the problems
- Low recall = the model misses important cases
- **In healthcare:** Low recall = missed diagnoses = patient harm

**The Tradeoff:** Precision vs Recall
- **High precision, low recall** = Conservative model (only fires when very sure, but misses cases)
- **High recall, low precision** = Aggressive model (flags everything, high false alarm rate)
- **Sweet spot** = Balanced precision & recall with good calibration

**Calibration:** "When the model says it's 80% confident, is it actually right 80% of the time?"
- Well-calibrated model = confidence scores are trustworthy
- Poorly calibrated (overconfident) = model says 90% confident but only right 70% of the time
- **In healthcare:** Overconfident models are dangerous because doctors rely on that confidence

Let's measure all of these.


In [ ]:
# Build confusion matrix for all suggestions where ground truth is available
# Filter: model fired + ground truth known
gt_df = model_df[model_df['ground_truth_correct'].notna()].copy()

print(f'Evaluating {len(gt_df):,} suggestions with ground truth labels')
print(f'Correct: {gt_df["ground_truth_correct"].sum():,}')
print(f'Incorrect: {(~gt_df["ground_truth_correct"]).sum():,}')

# Since we're looking at "model fired" suggestions, we consider:
# True Positive = fired AND correct
# False Positive = fired AND incorrect
# We don't have True Negatives (things not fired) in this dataset
y_true = gt_df['ground_truth_correct'].astype(int)
y_pred = (gt_df['model_fired'] & gt_df['suggestion_surfaced']).astype(int)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f'\n=== Classification Metrics ===')
print(f'Precision: {precision:.3f} ({tp} true pos / {tp + fp} predicted pos)')
print(f'Recall:    {recall:.3f} ({tp} true pos / {tp + fn} actual pos)')
print(f'F1 Score:  {f1:.3f}')
print(f'Accuracy:  {accuracy:.3f}')

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax,
            xticklabels=['Incorrect', 'Correct'], yticklabels=['Incorrect', 'Correct'],
            linewidths=2, linecolor='black', cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
ax.set_title(f'Confusion Matrix\nPrecision: {precision:.2%} | Recall: {recall:.2%} | F1: {f1:.3f}',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{OUTPUTS}04_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Confusion matrix saved')

In [ ]:
# Precision and recall by suggestion category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calculate metrics by category
category_metrics = {}
for cat in gt_df['suggestion_category'].unique():
    cat_df = gt_df[gt_df['suggestion_category'] == cat]
    y_true_cat = cat_df['ground_truth_correct'].astype(int)
    y_pred_cat = (cat_df['model_fired'] & cat_df['suggestion_surfaced']).astype(int)

    tp = ((y_true_cat == 1) & (y_pred_cat == 1)).sum()
    fp = ((y_true_cat == 0) & (y_pred_cat == 1)).sum()
    fn = ((y_true_cat == 1) & (y_pred_cat == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    category_metrics[cat] = {'precision': precision, 'recall': recall}

# Sort by precision
sorted_cats = sorted(category_metrics.items(), key=lambda x: x[1]['precision'], reverse=True)
cats = [c[0] for c in sorted_cats]
precisions = [c[1]['precision'] * 100 for c in sorted_cats]
recalls = [c[1]['recall'] * 100 for c in sorted_cats]

# Precision plot
ax = axes[0]
x = np.arange(len(cats))
bars = ax.bar(x, precisions, color=COLORS['blue'], alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Precision (%)', fontsize=11, fontweight='bold')
ax.set_title('Precision by Suggestion Category', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, f'{height:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=80, color='red', linestyle='--', linewidth=2, label='80% threshold', alpha=0.7)
ax.legend()

# Recall plot
ax = axes[1]
bars = ax.bar(x, recalls, color=COLORS['green'], alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Recall (%)', fontsize=11, fontweight='bold')
ax.set_title('Recall by Suggestion Category', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, f'{height:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=85, color='red', linestyle='--', linewidth=2, label='85% threshold', alpha=0.7)
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUTS}05_precision_recall_category.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nPrecision & Recall by Category:')
for cat in cats:
    print(f'{cat:25s}: Precision={category_metrics[cat]["precision"]:6.2%}  Recall={category_metrics[cat]["recall"]:6.2%}')
print('✓ Precision & recall visualization saved')

In [ ]:
# Confidence Calibration: is the model's confidence trustworthy?
# Only look at suggestions where confidence score is available
conf_df = gt_df[gt_df['confidence_score'].notna()].copy()

print(f'Calibrating on {len(conf_df):,} suggestions with confidence scores')

# Calculate calibration curve
prob_true, prob_pred = calibration_curve(conf_df['ground_truth_correct'].astype(int),
                                         conf_df['confidence_score'], n_bins=10, strategy='uniform')

# Plot calibration curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfectly Calibrated')
ax.plot(prob_pred, prob_true, 'o-', linewidth=2.5, markersize=8, color=COLORS['blue'],
        label='Model', markeredgecolor='black', markeredgewidth=1.5)
ax.fill_between(prob_pred, prob_true, prob_pred, alpha=0.2, color=COLORS['blue'], label='Calibration Gap')
ax.set_xlabel('Predicted Confidence', fontsize=11, fontweight='bold')
ax.set_ylabel('Actual Accuracy', fontsize=11, fontweight='bold')
ax.set_title('Confidence Calibration Curve', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(fontsize=10, loc='lower right')

# Add text annotation
calibration_error = np.abs(prob_pred - prob_true).mean()
ax.text(0.05, 0.95, f'Mean Calibration Error: {calibration_error:.3f}',
        transform=ax.transAxes, fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        verticalalignment='top')

# Confidence score distribution
ax = axes[1]
correct_conf = conf_df[conf_df['ground_truth_correct'] == True]['confidence_score']
incorrect_conf = conf_df[conf_df['ground_truth_correct'] == False]['confidence_score']

ax.hist(correct_conf, bins=30, alpha=0.6, label=f'Correct (n={len(correct_conf)})',
        color=COLORS['green'], edgecolor='black')
ax.hist(incorrect_conf, bins=30, alpha=0.6, label=f'Incorrect (n={len(incorrect_conf)})',
        color=COLORS['red'], edgecolor='black')
ax.set_xlabel('Confidence Score', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Distribution of Confidence Scores', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}06_calibration.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Confidence calibration error: {calibration_error:.3f}')
print(f'Mean confidence (correct): {correct_conf.mean():.3f}')
print(f'Mean confidence (incorrect): {incorrect_conf.mean():.3f}')
print('✓ Calibration visualization saved')

In [ ]:
# Model performance by cohort (time)
cohort_perf = []

for cohort in sorted(gt_df['signup_cohort'].unique()):
    cohort_df = gt_df[gt_df['signup_cohort'] == cohort]

    y_true_cohort = cohort_df['ground_truth_correct'].astype(int)
    y_pred_cohort = (cohort_df['model_fired'] & cohort_df['suggestion_surfaced']).astype(int)

    tp = ((y_true_cohort == 1) & (y_pred_cohort == 1)).sum()
    fp = ((y_true_cohort == 0) & (y_pred_cohort == 1)).sum()
    fn = ((y_true_cohort == 1) & (y_pred_cohort == 0)).sum()
    tn = ((y_true_cohort == 0) & (y_pred_cohort == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    cohort_perf.append({
        'cohort': cohort,
        'precision': precision,
        'recall': recall,
        'accuracy': accuracy,
        'n': len(cohort_df)
    })

perf_df = pd.DataFrame(cohort_perf)
cohort_order = sorted(perf_df['cohort'].unique())
perf_df['cohort_num'] = perf_df['cohort'].str.replace('W', '').astype(int)
perf_df = perf_df.sort_values('cohort_num')

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(perf_df['cohort'], perf_df['precision'] * 100, marker='o', linewidth=2.5,
        markersize=8, label='Precision', color=COLORS['blue'], markeredgecolor='black', markeredgewidth=1)
ax.plot(perf_df['cohort'], perf_df['recall'] * 100, marker='s', linewidth=2.5,
        markersize=8, label='Recall', color=COLORS['orange'], markeredgecolor='black', markeredgewidth=1)
ax.plot(perf_df['cohort'], perf_df['accuracy'] * 100, marker='^', linewidth=2.5,
        markersize=8, label='Accuracy', color=COLORS['green'], markeredgecolor='black', markeredgewidth=1)

ax.set_xlabel('Signup Cohort', fontsize=12, fontweight='bold')
ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Over Time (by Cohort)', fontsize=13, fontweight='bold', pad=20)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend(fontsize=11, loc='lower right')

# Rotate x labels
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}07_performance_by_cohort.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nModel Performance by Cohort:')
print(perf_df[['cohort', 'precision', 'recall', 'accuracy', 'n']].to_string(index=False))
print('\n✓ Cohort performance visualization saved')

## Section 3: Connecting Product Experience with Model Performance

**The real insight** comes from connecting the product experience layer with the model performance layer.

A suggestion that is **correct AND accepted** = value delivered.
A suggestion that is **incorrect AND accepted** = harm done.

Let's build a 2×2 matrix to understand where value and risk live:

| | Accepted | Rejected |
|---|---|---|
| **Correct** | ✅ VALUE CAPTURED | 😕 Missed Opportunity |
| **Incorrect** | ⚠️ FALSE POSITIVE COST | ✅ Good Rejection |

- **Value Captured:** User acted on a correct suggestion. This is the goal.
- **Missed Opportunity:** The model was right, but the user ignored it. Loss of potential value.
- **False Positive Cost:** User acted on an incorrect suggestion. This causes harm (in healthcare: wrong diagnosis acted upon).
- **Good Rejection:** User correctly rejected a bad suggestion. This prevents harm.

For AI products to succeed, you need:
1. **High model performance** (so most suggestions are correct)
2. **High product experience** (so users accept correct suggestions)
3. **Low false positive acceptance** (so users don't act on wrong suggestions)


In [ ]:
# Build the 2x2 matrix: correct/incorrect × accepted/rejected
# We need model_fired + ground_truth + user_accepted

outcome_df = model_df[model_df['ground_truth_correct'].notna()].copy()

# Create outcome categories
def categorize_outcome(row):
    correct = row['ground_truth_correct']
    accepted = row['user_accepted']

    if correct and accepted:
        return 'VALUE_CAPTURED'
    elif correct and not accepted:
        return 'MISSED_OPPORTUNITY'
    elif not correct and accepted:
        return 'FALSE_POSITIVE_COST'
    else:
        return 'GOOD_REJECTION'

outcome_df['outcome_category'] = outcome_df.apply(categorize_outcome, axis=1)

# Count outcomes
outcome_counts = outcome_df['outcome_category'].value_counts()
outcome_pcts = outcome_df['outcome_category'].value_counts(normalize=True) * 100

print('=== Outcome Matrix ===')
print(f'Value Captured:        {outcome_counts.get("VALUE_CAPTURED", 0):6,d} ({outcome_pcts.get("VALUE_CAPTURED", 0):6.1f}%)')
print(f'Missed Opportunity:    {outcome_counts.get("MISSED_OPPORTUNITY", 0):6,d} ({outcome_pcts.get("MISSED_OPPORTUNITY", 0):6.1f}%)')
print(f'False Positive Cost:   {outcome_counts.get("FALSE_POSITIVE_COST", 0):6,d} ({outcome_pcts.get("FALSE_POSITIVE_COST", 0):6.1f}%)')
print(f'Good Rejection:        {outcome_counts.get("GOOD_REJECTION", 0):6,d} ({outcome_pcts.get("GOOD_REJECTION", 0):6.1f}%)')

# Build 2x2 heatmap
matrix_data = np.zeros((2, 2))
matrix_data[0, 0] = outcome_counts.get('VALUE_CAPTURED', 0)  # Correct + Accepted
matrix_data[0, 1] = outcome_counts.get('MISSED_OPPORTUNITY', 0)  # Correct + Rejected
matrix_data[1, 0] = outcome_counts.get('FALSE_POSITIVE_COST', 0)  # Incorrect + Accepted
matrix_data[1, 1] = outcome_counts.get('GOOD_REJECTION', 0)  # Incorrect + Rejected

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matrix_data, annot=True, fmt=',.0f', cmap='RdYlGn', cbar=True, ax=ax,
            xticklabels=['Accepted', 'Rejected'], yticklabels=['Correct', 'Incorrect'],
            linewidths=3, linecolor='black', cbar_kws={'label': 'Count'},
            annot_kws={'fontsize': 14, 'fontweight': 'bold'})
ax.set_xlabel('User Action', fontsize=12, fontweight='bold')
ax.set_ylabel('Suggestion Correctness', fontsize=12, fontweight='bold')
ax.set_title('Outcome Matrix: Where Value & Risk Live', fontsize=13, fontweight='bold', pad=20)

# Add annotations
ax.text(0.5, -0.15, '✅ VALUE CAPTURED', transform=ax.transData, ha='center', fontsize=11,
        fontweight='bold', color='darkgreen', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
ax.text(1.5, -0.15, '😕 MISSED OPPORTUNITY', transform=ax.transData, ha='center', fontsize=11,
        fontweight='bold', color='darkorange', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax.text(0.5, 2.15, '⚠️ FALSE POSITIVE COST', transform=ax.transData, ha='center', fontsize=11,
        fontweight='bold', color='darkred', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
ax.text(1.5, 2.15, '✅ GOOD REJECTION', transform=ax.transData, ha='center', fontsize=11,
        fontweight='bold', color='darkgreen', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.savefig(f'{OUTPUTS}08_outcome_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Outcome matrix visualization saved')

In [ ]:
# Analyze outcome_value by acceptance and correctness
# Compare: value from correct accepted vs cost from incorrect accepted

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Value distribution by outcome type
ax = axes[0, 0]
outcome_values = []
outcome_labels = []
for outcome in ['VALUE_CAPTURED', 'MISSED_OPPORTUNITY', 'FALSE_POSITIVE_COST', 'GOOD_REJECTION']:
    values = outcome_df[outcome_df['outcome_category'] == outcome]['outcome_value'].dropna()
    if len(values) > 0:
        outcome_values.append(values)
        outcome_labels.append(outcome.replace('_', '\n'))

bp = ax.boxplot(outcome_values, labels=outcome_labels, patch_artist=True)
colors = ['green', 'orange', 'red', 'blue']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Outcome Value', fontsize=11, fontweight='bold')
ax.set_title('Outcome Value Distribution by Category', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

# Total net value by plan type
ax = axes[0, 1]
plan_value = outcome_df.groupby('plan_type')['outcome_value'].sum().sort_values(ascending=False)
bars = ax.bar(plan_value.index, plan_value.values,
              color=[COLORS['blue'], COLORS['orange'], COLORS['green']], alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Total Outcome Value', fontsize=11, fontweight='bold')
ax.set_title('Net Value Captured by Plan Type', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=2)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + max(plan_value)*0.02 if height > 0 else height - max(plan_value)*0.05,
            f'{height:,.0f}', ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Value over time (by cohort)
ax = axes[1, 0]
cohort_value = outcome_df.groupby('signup_cohort')['outcome_value'].sum().sort_index()
cohort_value_num = [int(x.replace('W', '')) for x in cohort_value.index]
ax.plot(range(len(cohort_value)), cohort_value.values, marker='o', linewidth=2.5, markersize=8,
        color=COLORS['blue'], markeredgecolor='black', markeredgewidth=1)
ax.fill_between(range(len(cohort_value)), cohort_value.values, alpha=0.3, color=COLORS['blue'])
ax.set_xticks(range(len(cohort_value)))
ax.set_xticklabels(cohort_value.index, rotation=45)
ax.set_ylabel('Total Outcome Value', fontsize=11, fontweight='bold')
ax.set_xlabel('Signup Cohort', fontsize=11, fontweight='bold')
ax.set_title('Net Value Captured Over Time', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
ax.grid(alpha=0.3)

# Value per accepted suggestion (correct vs incorrect)
ax = axes[1, 1]
accepted_correct = outcome_df[(outcome_df['user_accepted']) & (outcome_df['ground_truth_correct'])]['outcome_value']
accepted_incorrect = outcome_df[(outcome_df['user_accepted']) & (~outcome_df['ground_truth_correct'])]['outcome_value']

data_to_plot = [accepted_correct.dropna(), accepted_incorrect.dropna()]
labels_to_plot = [f'Correct\n(n={len(accepted_correct)})', f'Incorrect\n(n={len(accepted_incorrect)})']

bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)
colors_box = ['green', 'red']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Outcome Value per Suggestion', fontsize=11, fontweight='bold')
ax.set_title('Value Impact: Correct vs Incorrect Accepted', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}09_outcome_value.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nNet Outcome Value:')
print(f'Total (all users): {outcome_df["outcome_value"].sum():,.0f}')
print(f'By plan: {plan_value.to_dict()}')
print(f'Per user: {outcome_df.groupby("user_id")["outcome_value"].sum().mean():.2f}')
print('✓ Outcome value visualization saved')

In [ ]:
# Define and check guardrails for model performance and product health

print('=== MODEL PERFORMANCE GUARDRAILS ===\n')

guardrails = {}

# Guardrail 1: False Positive Rate
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
guardrail_1 = fpr < 0.25
guardrails['False Positive Rate'] = {'threshold': '<25%', 'actual': f'{fpr*100:.1f}%', 'status': 'PASS' if guardrail_1 else 'FAIL'}
print(f'1. False Positive Rate: {fpr*100:.1f}% (threshold: <25%) - {"✓ PASS" if guardrail_1 else "✗ FAIL"}')

# Guardrail 2: Calibration Error
guardrail_2 = calibration_error < 0.10
guardrails['Calibration Error'] = {'threshold': '<0.10', 'actual': f'{calibration_error:.3f}', 'status': 'PASS' if guardrail_2 else 'FAIL'}
print(f'2. Calibration Error: {calibration_error:.3f} (threshold: <0.10) - {"✓ PASS" if guardrail_2 else "✗ FAIL"}')

# Guardrail 3: Acceptance Rate
overall_acceptance = model_df['user_accepted'].mean()
guardrail_3 = overall_acceptance > 0.40
guardrails['Acceptance Rate'] = {'threshold': '>40%', 'actual': f'{overall_acceptance*100:.1f}%', 'status': 'PASS' if guardrail_3 else 'FAIL'}
print(f'3. Acceptance Rate: {overall_acceptance*100:.1f}% (threshold: >40%) - {"✓ PASS" if guardrail_3 else "✗ FAIL"}')

# Guardrail 4: Precision stays above 75%
guardrail_4 = precision > 0.75
guardrails['Precision'] = {'threshold': '>75%', 'actual': f'{precision*100:.1f}%', 'status': 'PASS' if guardrail_4 else 'FAIL'}
print(f'4. Precision: {precision*100:.1f}% (threshold: >75%) - {"✓ PASS" if guardrail_4 else "✗ FAIL"}')

# Guardrail 5: Recall stays above 80%
guardrail_5 = recall > 0.80
guardrails['Recall'] = {'threshold': '>80%', 'actual': f'{recall*100:.1f}%', 'status': 'PASS' if guardrail_5 else 'FAIL'}
print(f'5. Recall: {recall*100:.1f}% (threshold: >80%) - {"✓ PASS" if guardrail_5 else "✗ FAIL"}')

# Guardrail 6: False positive acceptance (of accepted suggestions, not too many are wrong)
false_pos_accepted = ((outcome_df['user_accepted']) & (~outcome_df['ground_truth_correct'])).sum()
total_accepted = outcome_df['user_accepted'].sum()
false_pos_accept_rate = false_pos_accepted / total_accepted if total_accepted > 0 else 0
guardrail_6 = false_pos_accept_rate < 0.20
guardrails['False Positive Acceptance'] = {'threshold': '<20%', 'actual': f'{false_pos_accept_rate*100:.1f}%', 'status': 'PASS' if guardrail_6 else 'FAIL'}
print(f'6. False Positive Acceptance Rate: {false_pos_accept_rate*100:.1f}% (threshold: <20%) - {"✓ PASS" if guardrail_6 else "✗ FAIL"}')

# Summary
total_guardrails = 6
passed = sum([g['status'] == 'PASS' for g in guardrails.values()])
print(f'\nOVERALL: {passed}/{total_guardrails} guardrails passed')

# Visualize guardrails
fig, ax = plt.subplots(figsize=(12, 6))
guardrail_names = list(guardrails.keys())
guardrail_status = [1 if guardrails[g]['status'] == 'PASS' else 0 for g in guardrail_names]
colors_status = ['green' if s == 1 else 'red' for s in guardrail_status]

bars = ax.barh(guardrail_names, [1]*len(guardrail_names), color=colors_status, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_xlim(0, 1)
ax.set_xlabel('')
ax.set_title('Model Performance Guardrails Status', fontsize=13, fontweight='bold', pad=20)
ax.set_xticks([])

for i, (name, g) in enumerate(guardrails.items()):
    status_text = f"{g['status']}: {g['actual']} (threshold: {g['threshold']})"
    ax.text(0.5, i, status_text, va='center', ha='center', fontweight='bold',
            fontsize=10, color='white', bbox=dict(boxstyle='round', facecolor='black', alpha=0.3))

plt.tight_layout()
plt.savefig(f'{OUTPUTS}10_guardrails.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ Guardrails visualization saved')

## Section 4: The North Star Metric for AI Products

For an AI product like SmarterDx, what single metric best captures success?

It's tempting to optimize for individual metrics:
- **Just optimize accuracy?** → You build a model that's right but nobody uses it
- **Just optimize acceptance?** → You build a UI users love but the AI is wrong
- **Just optimize precision?** → You become too conservative and miss important cases
- **Just optimize false positive rate?** → You suppress too many real signals

### The North Star: **Net Correct Suggestions Accepted per User per Month**

This metric combines BOTH layers:

$$\text{North Star} = \text{(Correct Suggestions Accepted)} - \text{(Incorrect Suggestions Accepted)}$$

**Why this works:**

1. ✅ **Only goes up when the model fires** (engagement)
2. ✅ **Only goes up when suggestions are correct** (model performance)
3. ✅ **Only goes up when users see and accept** (product experience)
4. ⚠️ **Goes DOWN when false positives are accepted** (penalizes harm)

This single metric forces alignment:
- Product team can't just increase acceptance through UI tricks (false positives pull it down)
- ML team can't just improve accuracy without user engagement (acceptance rate matters)
- Both teams must work together to capture value

**In practice at SmarterDx:**
- A correct diagnosis suggestion accepted = +1 (patient gets correct diagnosis)
- An incorrect diagnosis suggestion accepted = -1 (patient might get wrong diagnosis)
- A missed diagnosis (correct suggestion rejected) = 0 (no harm, but no value)
- A correct suggestion never shown = -1 (opportunity cost)

This metric is challenging but it's the truest measure of whether an AI product is creating value.


In [ ]:
# Calculate North Star: Net Correct Suggestions Accepted per User per Month

# Filter: ground truth available + user accepted
northstar_df = outcome_df[outcome_df['user_accepted']].copy()

# Count correct and incorrect accepted
correct_accepted = (northstar_df['ground_truth_correct']).sum()
incorrect_accepted = (~northstar_df['ground_truth_correct']).sum()
net_north_star = correct_accepted - incorrect_accepted

# Per user
users_accepting = northstar_df['user_id'].nunique()
north_star_per_user = net_north_star / users_accepting if users_accepting > 0 else 0

# Per month
months = (model_df['interaction_date'].max() - model_df['interaction_date'].min()).days / 30
north_star_per_month = net_north_star / months if months > 0 else 0

print('=== NORTH STAR METRIC ===')
print(f'Correct suggestions accepted: {correct_accepted:,}')
print(f'Incorrect suggestions accepted: {incorrect_accepted:,}')
print(f'Net North Star (total): {net_north_star:,}')
print(f'\nPer User: {north_star_per_user:.2f}')
print(f'Per Month: {north_star_per_month:.2f}')

# By plan type
print('\n=== North Star by Plan Type ===')
for plan in ['free', 'basic', 'pro']:
    plan_ns = northstar_df[northstar_df['plan_type'] == plan]
    plan_correct = plan_ns['ground_truth_correct'].sum()
    plan_incorrect = (~plan_ns['ground_truth_correct']).sum()
    plan_net = plan_correct - plan_incorrect
    plan_users = plan_ns['user_id'].nunique()
    plan_per_user = plan_net / plan_users if plan_users > 0 else 0
    print(f'{plan:8s}: {plan_net:6,d} total | {plan_per_user:6.2f} per user')

# By cohort
print('\n=== North Star by Cohort ===')
cohort_ns = northstar_df.groupby('signup_cohort').apply(
    lambda x: x['ground_truth_correct'].sum() - (~x['ground_truth_correct']).sum()
).sort_index()
print(cohort_ns.to_string())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall north star
ax = axes[0]
categories = ['Correct\nAccepted', 'Incorrect\nAccepted', 'Net North\nStar']
values = [correct_accepted, -incorrect_accepted, net_north_star]
colors_ns = ['green', 'red', 'blue']
bars = ax.bar(categories, values, color=colors_ns, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.axhline(y=0, color='black', linestyle='-', linewidth=2)
ax.set_ylabel('Count', fontsize=11, fontweight='bold')
ax.set_title('North Star Metric: Net Correct Suggestions Accepted', fontsize=12, fontweight='bold')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + max(abs(height) for height in values)*0.02,
            f'{int(height):,}', ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# North star by cohort
ax = axes[1]
cohort_ns_sorted = cohort_ns.sort_index()
x_pos = range(len(cohort_ns_sorted))
colors_cohort = ['green' if v >= 0 else 'red' for v in cohort_ns_sorted.values]
bars = ax.bar(x_pos, cohort_ns_sorted.values, color=colors_cohort, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(cohort_ns_sorted.index, rotation=45)
ax.set_ylabel('Net North Star', fontsize=11, fontweight='bold')
ax.set_xlabel('Signup Cohort', fontsize=11, fontweight='bold')
ax.set_title('North Star Metric by Cohort (Value Captured)', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}11_north_star.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ North star visualization saved')

## Section 5: Methods Comparison Reference

Below is a quick reference for when to use each metric in your analysis and in interviews:

| **Metric** | **Layer** | **What It Tells You** | **Use When** | **Limitations** |
|---|---|---|---|---|
| **AI Suggestion Funnel** | Product | % of users seeing & accepting AI | Adoption is slowing | Doesn't explain WHY users ignore |
| **Acceptance Rate** | Product | User trust in AI suggestions | Comparing segments | Can be high even if model is wrong |
| **Time-to-Decision** | Product | User confidence & UX friction | Decisions getting slower | Depends on domain (complex issues = slow) |
| **Precision** | Model | % of suggestions that are correct | Model performance is poor | Doesn't tell you if users see them |
| **Recall** | Model | % of real cases that model catches | You're missing important cases | Can be high while precision is low |
| **F1 Score** | Model | Balanced precision & recall | Need one number for model quality | Can hide important precision/recall tradeoffs |
| **Calibration** | Model | Are confidence scores trustworthy | Making decisions based on confidence | Requires ground truth (delayed in practice) |
| **Outcome Value** | Both | Net business impact of AI | Understanding true ROI | Depends on accurately measuring value/cost |
| **North Star** | Both | Best single metric for success | Aligning org around AI impact | Complex to measure, delayed feedback |

### Interview Tips for Each Metric

**"Tell me about model evaluation frameworks"**
- Start with: "I'd look at two layers: product experience and model performance"
- Mention precision/recall and the tradeoff
- Add: "But the real question is: do users trust it and does it help them?"

**"How would you detect when the model is degrading?"**
- Precision/recall by cohort over time (what you built in section 2)
- False positive rate increasing (what you built in guardrails)
- Acceptance rate declining (what you built in section 1)

**"Our model has 92% accuracy but users aren't adopting it. Why?"**
- "There's a gap between model performance and product experience. Let me investigate:"
  - Are suggestions being surfaced? (funnel)
  - Do users understand them? (time-to-decision)
  - Are they relevant to the user's workflow? (acceptance by segment)
  - Is the model overconfident? (calibration — maybe it says 92% confident but is actually 70%)

**"We want to improve the AI product. What's the one metric we should focus on?"**
- "I'd propose this North Star: net correct suggestions accepted per user per month"
- "It forces alignment because:"
  - Product team can't game it with UI tricks
  - ML team can't just improve accuracy
  - Both teams must work together
  - It captures business impact, not just technical metrics


In [ ]:
# Summary of key findings
print('='*60)
print('KEY FINDINGS: Model Performance & Outcome Impact Analysis')
print('='*60)

print('\n📊 PRODUCT EXPERIENCE LAYER')
print(f'  • {model_df["model_fired"].mean()*100:.1f}% of interactions trigger model')
print(f'  • {(model_df["suggestion_surfaced"].sum() / model_df["model_fired"].sum())*100:.1f}% of fired suggestions reach users')
print(f'  • {model_df["user_accepted"].mean()*100:.1f}% overall acceptance rate')
print(f'  • Pro users accept {model_df[model_df["plan_type"]=="pro"]["user_accepted"].mean()*100:.1f}% vs Free {model_df[model_df["plan_type"]=="free"]["user_accepted"].mean()*100:.1f}%')

print('\n🧠 MODEL PERFORMANCE LAYER')
print(f'  • Precision: {precision*100:.1f}% (when model speaks, it\'s right {int(precision*100)}% of time)')
print(f'  • Recall: {recall*100:.1f}% (catches {int(recall*100)}% of real cases)')
print(f'  • F1 Score: {f1:.3f}')
print(f'  • Calibration Error: {calibration_error:.3f} (confidence scores are {"well-calibrated" if calibration_error < 0.10 else "overconfident"})')

print('\n🎯 OUTCOME IMPACT')
print(f'  • Value Captured: {outcome_counts.get("VALUE_CAPTURED", 0):,} correct suggestions accepted')
print(f'  • False Positives Accepted: {outcome_counts.get("FALSE_POSITIVE_COST", 0):,} incorrect suggestions accepted')
print(f'  • Net Outcome Value: {net_north_star:,}')
print(f'  • North Star: {north_star_per_user:.2f} per user')

print('\n✅ GUARDRAILS STATUS')
passed_guardrails = sum([g['status'] == 'PASS' for g in guardrails.values()])
print(f'  • {passed_guardrails}/6 guardrails passing')
for name, g in guardrails.items():
    status_emoji = '✓' if g['status'] == 'PASS' else '✗'
    print(f'    {status_emoji} {name}: {g["actual"]} (threshold: {g["threshold"]})')

print('\n💡 RECOMMENDATIONS')
if guardrails['False Positive Rate']['status'] != 'PASS':
    print('  1. False positive rate too high - review confidence threshold tuning')
if guardrails['Acceptance Rate']['status'] != 'PASS':
    print('  1. Acceptance rate below 40% - investigate UX friction or model relevance')
if guardrails['Calibration Error']['status'] != 'PASS':
    print('  1. Model is overconfident - retrain with calibration loss')
if guardrails['Precision']['status'] != 'PASS':
    print('  1. Precision below 75% - raise confidence threshold or improve model')
if guardrails['Recall']['status'] != 'PASS':
    print('  1. Recall below 80% - lower confidence threshold or improve model coverage')

print('\n' + '='*60)
print('✓ Analysis complete - all outputs saved to ' + OUTPUTS)
print('='*60)

## Section 6: Conclusion & Interview Takeaways

### What We Learned

This notebook demonstrated a complete framework for evaluating AI products through two interconnected lenses:

**Product Experience Layer** shows whether users are engaging with the AI:
- Funnel analysis reveals where the adoption pipeline breaks
- Acceptance rates by segment show who trusts the AI
- Time-to-decision indicates confidence and UX friction

**Model Performance Layer** shows whether the AI is actually accurate:
- Precision & recall measure the classic accuracy-coverage tradeoff
- Confusion matrix gives concrete counts of errors
- Calibration reveals whether confidence scores are trustworthy
- Performance over cohorts shows whether model improvements are reaching users

**Connecting Both Layers** is where real insights emerge:
- The 2×2 outcome matrix shows where value and risk live
- Net outcome value quantifies the true business impact
- Guardrails protect against degradation
- The North Star metric aligns the entire organization

### For Your Interview

If you're interviewing for a Senior Product Analyst role at an AI company like SmarterDx:

1. **Lead with the two-layer framework** — It shows sophisticated thinking about AI products
2. **Use concrete metrics** — Don't just say "measure precision"; show how and why
3. **Connect to business impact** — "Here's the model performance... and here's what users actually do with it... and here's the net value."
4. **Emphasize guardrails** — "I wouldn't just monitor accuracy. I'd also monitor false positive rate because..."
5. **Think about tradeoffs** — "High recall or high precision? That depends on the cost of false negatives vs false positives."
6. **Propose actionable next steps** — "If this guardrail fails, here's what I'd investigate..."

### Final Thought

The best AI products aren't built by ML teams optimizing models in isolation, or product teams optimizing UI in isolation. They're built by **teams that measure both**, understand the connection, and align around metrics that capture true value creation.

That's what this notebook demonstrates. Use it in your interview.

---

**Repository**: Product Analytics Interview Prep
**Project**: SmarterDx AI Product Analysis
**Date**: 2026-04-05
**Analyst**: Trinidad Cisneros
